# TelecomX LATAM – Data Cleaning and Churn Analysis

This notebook performs:
- Data extraction from JSON
- Data cleaning and transformation
- Exploratory Data Analysis (EDA)
- Correlation analysis
- Basic churn metrics

Generated automatically for improvement and best practices.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import json_normalize

## 1. Extract Data

In [ ]:
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

response = requests.get(url)
response.raise_for_status()

data_json = response.json()

df = json_normalize(data_json)

df.head()

## 2. Clean Column Names

In [ ]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(".", "_", regex=False)
)

df.columns

## 3. Clean Text Fields

In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()

df.head()

## 4. Convert Binary Variables

In [ ]:
binary_map = {
    'yes': 1,
    'no': 0,
    'no internet service': 0,
    'no phone service': 0
}

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].replace(binary_map)

## 5. Convert Numeric Fields

In [ ]:
if 'account_charges_total' in df.columns:
    df['account_charges_total'] = pd.to_numeric(df['account_charges_total'], errors='coerce')

## 6. Feature Engineering

In [ ]:
if 'account_charges_monthly' in df.columns:
    df['daily_charges'] = df['account_charges_monthly'] / 30

## 7. Churn Distribution

In [ ]:
if 'churn' in df.columns:
    churn_counts = df['churn'].value_counts(dropna=False)
    churn_counts

## 8. Churn Rate

In [ ]:
if 'churn' in df.columns:
    churn_rate = df['churn'].mean() * 100
    print(f"Churn rate: {churn_rate:.2f}%")

## 9. Correlation Matrix

In [ ]:
df_numeric = df.select_dtypes(include=[np.number])

corr_matrix = df_numeric.dropna().corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)

plt.title('Correlation Matrix')
plt.show()